# Colab Setup & TabFM Installation

In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!git clone --quiet https://github.com/google-research/tabfm.git /content/tabfm
%cd /content/tabfm
!git checkout --quiet 603f4dd1312d8cb80eb7cf92e17287b3bceaa8d8
!pip install -e .[pytorch] -q
print("installed")

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 0 MiB
/content/tabfm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 5.3 MB/s eta 0:00:00
  Building editable for tabfm (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inflect 7.5.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.
installed


# Load Exchange Dataset

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd

EXCHANGE_PATH = "/content/drive/MyDrive/tabfm_input.csv"

df_x = pd.read_csv(EXCHANGE_PATH)
meta = ["__row_id", "__y", "__gender", "__split"]
features = [c for c in df_x.columns if c not in meta]

for col in features:
    if df_x[col].dtype == object:
        df_x[col] = df_x[col].astype("category")

is_train, is_test = df_x["__split"] == "train", df_x["__split"] == "test"
X_train, y_train = df_x.loc[is_train, features], df_x.loc[is_train, "__y"].to_numpy()
X_test = df_x.loc[is_test, features]

n_cat = sum(str(df_x[c].dtype) == "category" for c in features)
print("train:", X_train.shape, "| test:", X_test.shape)
print("categorical:", n_cat, "| numeric:", len(features) - n_cat)
print("first 3 test row ids:", df_x.loc[is_test, "__row_id"].head(3).tolist())

Mounted at /content/drive
train: (600, 18) | test: (200, 18)
categorical: 11 | numeric: 7
first 3 test row ids: [2, 14, 17]


# Load TabFM Checkpoint

In [3]:
import time
import torch
from tabfm import TabFMClassifier
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0

t0 = time.perf_counter()
model = tabfm_v1_0_0.load(model_type="classification")
print(f"checkpoint loaded in {time.perf_counter() - t0:.1f}s")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print("device     :", device)
print("params     :", f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print("max_classes:", model.max_classes)
if device == "cuda":
    print(f"VRAM used  : {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights from local directory
checkpoint loaded in 116.4s
device     : cuda
params     : 1.64B
max_classes: 10
VRAM used  : 3.05 GB


# TabFM Smoke Test

In [4]:
import numpy as np

clf = TabFMClassifier(model=model, n_estimators=4, random_state=42, batch_size=1)

t0 = time.perf_counter(); clf.fit(X_train, y_train); fit_s = time.perf_counter() - t0
t0 = time.perf_counter(); p_smoke = clf.predict_proba(X_test.iloc[:20]); smoke_s = time.perf_counter() - t0

print("classes_     :", clf.classes_)
print("proba shape  :", p_smoke.shape)
print("sums to 1    :", np.allclose(p_smoke.sum(axis=1), 1.0))
print(f"fit {fit_s:.1f}s | 20 rows {smoke_s:.1f}s -> ~{smoke_s / 20 * 200:.0f}s for 200 rows")
print("sample probs :", np.round(p_smoke[:3], 4).tolist())

classes_     : [0 1]
proba shape  : (20, 2)
sums to 1    : True
fit 0.1s | 20 rows 5.3s -> ~53s for 200 rows
sample probs : [[0.9115999937057495, 0.08839999884366989], [0.22949999570846558, 0.7705000042915344], [0.4246000051498413, 0.5753999948501587]]


In [6]:

import json
 
  # n_estimators=32 is TabFM's documented default and what its published benchmarks use.
  # cache_context left off: it int8-quantizes the cached K/V by default, and I would rather
  # report exact numerics than save a minute.
  clf = TabFMClassifier(model=model, n_estimators=32, random_state=42, batch_size=1)
 
  t0 = time.perf_counter(); clf.fit(X_train, y_train); fit_s = time.perf_counter() - t0
  t0 = time.perf_counter(); proba_full = clf.predict_proba(X_test); predict_s = time.perf_counter() - t0
 
  assert list(clf.classes_) == [0, 1], clf.classes_
  assert proba_full.shape == (len(X_test), 2)
  assert np.allclose(proba_full.sum(axis=1), 1.0)
 
  out = pd.DataFrame({
      "__row_id": df_x.loc[is_test, "__row_id"].to_numpy(),
      "proba_default": proba_full[:, 1],      # class 1 = default
  })
  out.to_csv("/content/drive/MyDrive/tabfm_predictions_german.csv", index=False)
 
  meta = {
      "dataset": "german_credit",
      "track": "T4_tabfm",
      "tabfm_commit": "603f4dd1312d8cb80eb7cf92e17287b3bceaa8d8",
      "checkpoint": "google/tabfm-1.0.0-pytorch/classification",
      "backend": "pytorch",
      "device": device,
      "n_params": int(sum(p.numel() for p in model.parameters())),
      "n_estimators": 32,
      "random_state": 42,
      "cache_context": False,
      "n_train_context_rows": int(len(X_train)),
      "n_test_rows": int(len(X_test)),
      "fit_seconds": round(fit_s, 3),
      "predict_seconds_total": round(predict_s, 3),
      "predict_seconds_per_row": round(predict_s / len(X_test), 4),
      "vram_gb_after_predict": round(torch.cuda.memory_allocated() / 1024**3, 2) if device == "cuda" else None,
  }
  with open("/content/drive/MyDrive/tabfm_meta_german.json", "w") as f:
      json.dump(meta, f, indent=2)
 
  print(json.dumps(meta, indent=2))
  print("\nproba_default: min %.4f  mean %.4f  max %.4f" % (out.proba_default.min(), out.proba_default.mean(),
  out.proba_default.max()))

{
  "dataset": "german_credit",
  "track": "T4_tabfm",
  "tabfm_commit": "603f4dd1312d8cb80eb7cf92e17287b3bceaa8d8",
  "checkpoint": "google/tabfm-1.0.0-pytorch/classification",
  "backend": "pytorch",
  "device": "cuda",
  "n_params": 1639444298,
  "n_estimators": 32,
  "random_state": 42,
  "cache_context": false,
  "n_train_context_rows": 600,
  "n_test_rows": 200,
  "fit_seconds": 0.074,
  "predict_seconds_total": 53.233,
  "predict_seconds_per_row": 0.2662,
  "vram_gb_after_predict": 3.06
}

proba_default: min 0.0147  mean 0.3105  max 0.8969
